# Notebook D Category Validation

Validation-only notebook for the current best Nemotron LoRA submission.

Current best: Version 15, `adapter_sft_v2_bit_bal128_asstloss`, public score `0.56`.

This notebook does not submit to the leaderboard. It loads Notebook C `submission.zip`, runs vLLM validation on `train.csv`, reports category-level accuracy, and writes CSV diagnostics under `/kaggle/working/category_validation`.


In [ ]:
from pathlib import Path
import os

RUN_EVALUATION = True
EVALUATION_SAMPLE_SIZE = 950
RANDOM_SEED = 42

ADAPTER_DIR = Path('/kaggle/working/adapter_for_validation_d')
DEBUG_DIR = Path('/kaggle/working/category_validation')
MISTAKES_DIR = DEBUG_DIR / 'mistakes'

COMPETITION_DIR = Path('/kaggle/input/nvidia-nemotron-3-reasoning-challenge')
MODEL_PATH = Path('/kaggle/input/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
SUBMISSION_ZIP_PATH = None

DEBUG_DIR.mkdir(parents=True, exist_ok=True)
MISTAKES_DIR.mkdir(parents=True, exist_ok=True)

print('RUN_EVALUATION:', RUN_EVALUATION)
print('EVALUATION_SAMPLE_SIZE:', EVALUATION_SAMPLE_SIZE)
print('ADAPTER_DIR:', ADAPTER_DIR)
print('DEBUG_DIR:', DEBUG_DIR)


In [ ]:
import glob
import json
import shutil
import zipfile
from pathlib import Path


def discover_submission_zip():
    search_roots = [
        Path('/kaggle/input/notebooks'),
        Path('/kaggle/input'),
        Path('/kaggle/working'),
    ]
    candidates = []
    for root in search_roots:
        if root.exists():
            candidates.extend(sorted(root.glob('**/submission.zip')))
    if not candidates:
        raise FileNotFoundError('Could not find submission.zip under /kaggle/input/notebooks, /kaggle/input, or /kaggle/working')
    for path in candidates:
        if 'notebook-c-adapter-validation-submission-pack-tr' in str(path):
            return path
    return candidates[0]


SUBMISSION_ZIP_PATH = discover_submission_zip()
print('SUBMISSION_ZIP_PATH:', SUBMISSION_ZIP_PATH)

if ADAPTER_DIR.exists():
    shutil.rmtree(ADAPTER_DIR)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(SUBMISSION_ZIP_PATH, 'r') as zf:
    zf.extractall(ADAPTER_DIR)
    print('Zip contents:', zf.namelist())

cfg_candidates = sorted(ADAPTER_DIR.glob('**/adapter_config.json'))
assert cfg_candidates, 'adapter_config.json missing after extracting submission.zip'

adapter_root = cfg_candidates[0].parent
if adapter_root != ADAPTER_DIR:
    for item in adapter_root.iterdir():
        dst = ADAPTER_DIR / item.name
        if dst.exists():
            continue
        if item.is_dir():
            shutil.copytree(item, dst)
        else:
            shutil.copy2(item, dst)

cfg_path = ADAPTER_DIR / 'adapter_config.json'
safetensors_path = ADAPTER_DIR / 'adapter_model.safetensors'
bin_path = ADAPTER_DIR / 'adapter_model.bin'

assert cfg_path.exists(), f'Missing {cfg_path}'
assert safetensors_path.exists() or bin_path.exists(), 'Missing adapter_model.safetensors or adapter_model.bin'

with cfg_path.open() as f:
    adapter_cfg = json.load(f)

rank = int(adapter_cfg.get('r', adapter_cfg.get('rank', -1)))
assert 0 < rank <= 32, f'Unexpected LoRA rank: {rank}'

print('Adapter path:', ADAPTER_DIR)
print('Detected LoRA rank:', rank)
print('target_modules:', adapter_cfg.get('target_modules'))
print('Adapter files present:', sorted(p.name for p in ADAPTER_DIR.iterdir() if p.is_file()))


In [ ]:
import pandas as pd
from pathlib import Path


def find_train_csv():
    direct = Path('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')
    if direct.exists():
        return direct
    matches = sorted(Path('/kaggle/input').glob('**/train.csv'))
    if not matches:
        raise FileNotFoundError('Could not find train.csv under /kaggle/input')
    return matches[0]


TRAIN_CSV = find_train_csv()
train_df = pd.read_csv(TRAIN_CSV)
print('TRAIN_CSV:', TRAIN_CSV)
print('train shape:', train_df.shape)
print('columns:', list(train_df.columns))

prompt_col = 'prompt' if 'prompt' in train_df.columns else 'problem'
answer_col = 'answer'
id_col = 'id' if 'id' in train_df.columns else None
assert prompt_col in train_df.columns, 'No prompt/problem column found'
assert answer_col in train_df.columns, 'No answer column found'


In [ ]:
import re

OPERATOR_RE = re.compile(r'(?:operator|operation|rule)\s*[:=]?\s*([+\-*/^=<>]+|[A-Za-z_][A-Za-z0-9_]*)', re.IGNORECASE)
QUERY_OP_RE = re.compile(r'(?:query|question|find|solve).*?([+\-*/^=<>]+|[A-Za-z_][A-Za-z0-9_]*)', re.IGNORECASE | re.DOTALL)


def _examples_text(prompt: str) -> str:
    lower = prompt.lower()
    cut_points = [lower.find(marker) for marker in ['query', 'question', 'now solve', 'find the'] if lower.find(marker) >= 0]
    if not cut_points:
        return prompt
    return prompt[:min(cut_points)]


def _query_operator(prompt: str):
    m = QUERY_OP_RE.search(prompt)
    if m:
        return m.group(1)
    tail = prompt[-800:]
    ops = re.findall(r'([+\-*/^=<>]+)', tail)
    return ops[-1] if ops else None


def _operator_appears_in_examples(prompt: str) -> bool:
    query_op = _query_operator(prompt)
    if not query_op:
        return False
    return query_op in _examples_text(prompt)


def detect_category(prompt):
    text = str(prompt)
    lower = text.lower()
    if 'secret bit manipulation rule transforms 8-bit binary numbers' in lower:
        return 'bit_manipulation'
    if 'secret encryption rules are used on text' in lower:
        return 'cipher'
    if 'secret set of transformation rules is applied to equations' in lower:
        examples = _examples_text(text)
        has_digits = bool(re.search(r'\d', examples))
        op_seen = _operator_appears_in_examples(text)
        if has_digits:
            return 'equation_numeric_deduce' if op_seen else 'equation_numeric_guess'
        return 'cryptarithm_deduce' if op_seen else 'cryptarithm_guess'
    if 'gravitational constant has been secretly changed' in lower:
        return 'gravity'
    if 'converted into a different numeral system' in lower:
        return 'numeral'
    if 'secret unit conversion is applied to measurements' in lower:
        return 'unit_conversion'
    return 'unknown'


train_df['category'] = train_df[prompt_col].map(detect_category)
print('Category distribution:')
print(train_df['category'].value_counts(dropna=False).to_string())


In [ ]:
import math
import pandas as pd


def stratified_sample(df, category_col='category', total=950, seed=42):
    groups = {cat: g for cat, g in df.groupby(category_col, dropna=False)}
    if len(df) <= total:
        return df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    categories = sorted(groups.keys(), key=lambda x: str(x))
    base = max(1, total // max(1, len(categories)))
    parts = []
    remaining_groups = []
    used = 0
    for cat in categories:
        g = groups[cat]
        take = min(len(g), base)
        parts.append(g.sample(n=take, random_state=seed))
        used += take
        if len(g) > take:
            remaining_groups.append((cat, g.drop(parts[-1].index)))
    remaining = total - used
    if remaining > 0 and remaining_groups:
        pool = pd.concat([g for _, g in remaining_groups])
        take = min(remaining, len(pool))
        if take > 0:
            parts.append(pool.sample(n=take, random_state=seed + 1))
    return pd.concat(parts).sample(frac=1.0, random_state=seed + 2).reset_index(drop=True)


sample_df = stratified_sample(train_df, total=EVALUATION_SAMPLE_SIZE, seed=RANDOM_SEED)
print('Sample size:', len(sample_df))
print('Sample category distribution:')
print(sample_df['category'].value_counts(dropna=False).to_string())


In [ ]:
import os
import sys
import shutil
import shlex
import subprocess
from pathlib import Path

os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'


def _contains_vllm(root: Path) -> bool:
    return (root / 'vllm').exists() or any(root.glob('**/vllm/__init__.py'))


def find_metric_utility_root() -> Path:
    candidates = [
        Path('/kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script'),
        Path('/kaggle/input/notebooks/metric/nvidia-metric-utility-script'),
        Path('/kaggle/input/notebooks/metric/nvidia_metric_utility_script'),
    ]
    candidates.extend(sorted(Path('/kaggle/input').glob('**/nvidia*metric*utility*')))

    seen = set()
    for root in candidates:
        root = Path(root)
        key = str(root)
        if key in seen:
            continue
        seen.add(key)
        if root.exists() and root.is_dir():
            return root

    ryan_root = Path('/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')
    if ryan_root.exists() and _contains_vllm(ryan_root):
        return ryan_root

    checked = [str(p) for p in candidates] + [str(ryan_root)]
    raise FileNotFoundError(
        'vLLM utility root not found. Add the NVIDIA metric utility script as a Notebook D input/source. '
        'Checked: ' + ', '.join(checked)
    )


UTILITY_ROOT = find_metric_utility_root()
print('Selected utility root:', UTILITY_ROOT)

copy_cmd = f'tar -cf - -C {shlex.quote(str(UTILITY_ROOT))} . | tar -xf - -C /tmp'
print('Copying utility root to /tmp')
subprocess.run(copy_cmd, shell=True, check=True)

for exe in [
    Path('/tmp/triton/backends/nvidia/bin/ptxas'),
    Path('/tmp/triton/backends/nvidia/bin/ptxas-blackwell'),
]:
    if exe.exists():
        exe.chmod(0o755)
        print('chmod +x:', exe)
    else:
        print('WARN: utility executable missing:', exe)

if '/tmp' not in sys.path:
    sys.path.insert(0, '/tmp')

triton_ptxas = Path('/tmp/triton/backends/nvidia/bin/ptxas')
if triton_ptxas.exists():
    os.environ['TRITON_PTXAS_PATH'] = str(triton_ptxas)
    print('TRITON_PTXAS_PATH:', os.environ['TRITON_PTXAS_PATH'])

triton_blackwell = Path('/tmp/triton/backends/nvidia/bin/ptxas-blackwell')
if triton_blackwell.exists():
    os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = str(triton_blackwell)
    print('TRITON_PTXAS_BLACKWELL_PATH:', os.environ['TRITON_PTXAS_BLACKWELL_PATH'])

print('/tmp/vllm exists:', Path('/tmp/vllm').exists())
print('Bootstrap complete')


In [ ]:
from pathlib import Path


def resolve_model_path():
    if MODEL_PATH.exists():
        return str(MODEL_PATH)
    matches = sorted(Path('/kaggle/input').glob('**/nemotron-3-nano-30b-a3b-bf16/**/default'))
    if matches:
        return str(matches[0])
    matches = sorted(Path('/kaggle/input').glob('**/transformers/default'))
    if matches:
        return str(matches[0])
    raise FileNotFoundError('Could not find Nemotron model path under /kaggle/input')


MODEL_DIR = resolve_model_path()
print('MODEL_DIR:', MODEL_DIR)


In [ ]:
if RUN_EVALUATION:
    from transformers import AutoTokenizer
    try:
        from vllm import LLM, SamplingParams
        from vllm.lora.request import LoRARequest
    except ModuleNotFoundError as exc:
        if exc.name == 'vllm':
            raise ModuleNotFoundError(
                'vLLM not found. Add the NVIDIA metric utility script as a Notebook D input/source.'
            ) from exc
        raise

    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True, local_files_only=True)

    llm_kwargs = dict(
        model=MODEL_DIR,
        trust_remote_code=True,
        enable_lora=True,
        max_lora_rank=32,
        max_model_len=8192,
        gpu_memory_utilization=0.85,
        dtype='bfloat16',
    )
    try:
        llm = LLM(max_num_seqs=64, **llm_kwargs)
        MAX_NUM_SEQS = 64
    except Exception as exc:
        print('WARN: max_num_seqs=64 failed, retrying with 16:', repr(exc))
        llm = LLM(max_num_seqs=16, **llm_kwargs)
        MAX_NUM_SEQS = 16

    sampling_params = SamplingParams(
        max_tokens=7680,
        temperature=0.0,
        top_p=1.0,
    )
    lora_request = LoRARequest('adapter', 1, str(ADAPTER_DIR))
    print('Loaded vLLM with LoRA. MAX_NUM_SEQS:', MAX_NUM_SEQS)
else:
    print('RUN_EVALUATION=False; skipped vLLM load')


In [ ]:
FINAL_INSTRUCTION = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'


def build_chat_prompt(prompt):
    user_content = str(prompt).rstrip() + FINAL_INSTRUCTION
    messages = [{'role': 'user', 'content': user_content}]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )


if RUN_EVALUATION:
    prompts = [build_chat_prompt(p) for p in sample_df[prompt_col].tolist()]
    print('Prepared prompts:', len(prompts))
else:
    prompts = []


In [ ]:
if RUN_EVALUATION:
    outputs = llm.generate(prompts, sampling_params=sampling_params, lora_request=lora_request)
    raw_outputs = [out.outputs[0].text if out.outputs else '' for out in outputs]
    print('Generated outputs:', len(raw_outputs))
else:
    raw_outputs = []


In [ ]:
import math
import re
from decimal import Decimal, InvalidOperation

BOXED_RE = re.compile(r'\\boxed\{')


def extract_braced(text, start):
    depth = 0
    chars = []
    i = start
    while i < len(text):
        ch = text[i]
        if ch == '{':
            depth += 1
            if depth > 1:
                chars.append(ch)
        elif ch == '}':
            depth -= 1
            if depth == 0:
                return ''.join(chars).strip()
            chars.append(ch)
        else:
            chars.append(ch)
        i += 1
    return None


def extract_final_answer(raw):
    text = '' if raw is None else str(raw)
    matches = list(BOXED_RE.finditer(text))
    if matches:
        value = extract_braced(text, matches[-1].end() - 1)
        if value is not None:
            return value.strip(), True
    # Fallback roughly mimics the competition style: take the last non-empty line/token.
    lines = [ln.strip() for ln in text.strip().splitlines() if ln.strip()]
    if not lines:
        return '', False
    last = lines[-1]
    last = re.sub(r'^(final answer|answer)\s*[:=]\s*', '', last, flags=re.IGNORECASE).strip()
    return last.strip('` $'), False


def normalize_text(value):
    return re.sub(r'\s+', ' ', str(value).strip()).lower()


def parse_number(value):
    s = str(value).strip().replace(',', '')
    s = s.strip('$')
    try:
        return float(Decimal(s))
    except (InvalidOperation, ValueError):
        return None


def verify(answer, prediction):
    a = str(answer).strip()
    p = str(prediction).strip()
    if re.fullmatch(r'[01]+', a) and re.fullmatch(r'[01]+', p):
        return a == p
    a_num = parse_number(a)
    p_num = parse_number(p)
    if a_num is not None and p_num is not None:
        return math.isclose(a_num, p_num, rel_tol=1e-2, abs_tol=1e-5)
    return normalize_text(a) == normalize_text(p)


In [ ]:
import pandas as pd

if RUN_EVALUATION:
    extracted = [extract_final_answer(raw) for raw in raw_outputs]
    predictions = [x[0] for x in extracted]
    has_boxed = [x[1] for x in extracted]

    pred_df = sample_df.copy()
    pred_df['prediction'] = predictions
    pred_df['has_boxed'] = has_boxed
    pred_df['raw_output'] = raw_outputs
    pred_df['correct'] = [verify(a, p) for a, p in zip(pred_df[answer_col], pred_df['prediction'])]
else:
    pred_df = sample_df.copy()
    pred_df['prediction'] = ''
    pred_df['has_boxed'] = False
    pred_df['raw_output'] = ''
    pred_df['correct'] = False

out_cols = []
if id_col:
    out_cols.append(id_col)
out_cols.extend(['category', answer_col, 'prediction', 'correct', 'has_boxed', 'raw_output'])
validation_predictions = pred_df[out_cols].rename(columns={id_col or 'index': 'id', answer_col: 'answer'})
if 'id' not in validation_predictions.columns:
    validation_predictions.insert(0, 'id', pred_df.index)

category_summary = (
    pred_df.groupby('category', dropna=False)['correct']
    .agg(['sum', 'count'])
    .reset_index()
    .rename(columns={'sum': 'correct', 'count': 'total'})
)
category_summary['accuracy'] = category_summary['correct'] / category_summary['total']
category_summary['contribution'] = category_summary['correct'] / len(pred_df)
category_summary = category_summary.sort_values(['accuracy', 'total'], ascending=[True, False])

boxed_summary = (
    pred_df.groupby(['category', 'has_boxed'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values(['category', 'has_boxed'])
)

validation_predictions.to_csv(DEBUG_DIR / 'validation_predictions.csv', index=False)
category_summary.to_csv(DEBUG_DIR / 'category_summary.csv', index=False)
boxed_summary.to_csv(DEBUG_DIR / 'boxed_summary.csv', index=False)

for category, group in pred_df.loc[~pred_df['correct']].groupby('category', dropna=False):
    safe = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(category)) or 'unknown'
    cols = out_cols if all(c in group.columns for c in out_cols) else group.columns.tolist()
    group[cols].rename(columns={answer_col: 'answer'}).to_csv(MISTAKES_DIR / f'{safe}.csv', index=False)

print('Adapter path:', ADAPTER_DIR)
print('Sample size:', len(pred_df))
print('Category distribution:')
print(pred_df['category'].value_counts(dropna=False).to_string())
print('Overall accuracy:', float(pred_df['correct'].mean()) if len(pred_df) else 0.0)
print('Category summary:')
print(category_summary.to_string(index=False))
print('Boxed-answer failure table:')
print(boxed_summary.to_string(index=False))

worst_categories = category_summary.head(3)['category'].tolist()
for category in worst_categories:
    wrong = pred_df[(pred_df['category'] == category) & (~pred_df['correct'])].head(3)
    if wrong.empty:
        continue
    print('\nTop raw mistakes for category:', category)
    for _, row in wrong.iterrows():
        rid = row[id_col] if id_col else row.name
        print('id:', rid, 'answer:', row[answer_col], 'prediction:', row['prediction'], 'has_boxed:', row['has_boxed'])
        print(str(row['raw_output'])[:1200].replace('\n', ' '))


In [ ]:
assert (DEBUG_DIR / 'validation_predictions.csv').exists(), 'validation_predictions.csv missing'
assert (DEBUG_DIR / 'category_summary.csv').exists(), 'category_summary.csv missing'
print('Notebook D category validation complete.')
